# 09 — Euro 2012 Analytics: Multi-Axis Slicing & Metric Filtering
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Sports Analytics, and Data Manipulation Interviews.*

---

## 📌 Executive Summary & Interview Expectations
This lab focuses on **2D positional slicing (`iloc`)**, **vectorized string filtering**, **multi-column sorting**, and **metric extraction**. In technical interviews, interviewers use datasets like this to test whether you can slice tabular matrices without hardcoding brittle indices.

### Core Competencies Tested in this Module:
1. **Positional Matrix Slicing (`iloc`)**: Negative indexing `iloc[:, :-3]` and coordinate bounds.
2. **Multi-Key Sorting**: Sorting by multiple metrics with priority rankings (`by=['Red Cards', 'Yellow Cards']`).
3. **Prefix String Filtering**: Using `.str.startswith()` safely.
4. **Fixing Hardcoded Index Lookups**: Replacing fragile coordinate indexing (`loc[3, ...]`) with dynamic label-based filtering (`df['Team'].isin(...)`).
5. **Interview Corner**: Negative slice performance, regex vs string methods, and conversion efficiency metrics.

## 1. Environment Setup & Data Ingestion

In [1]:
import os
import numpy as np
import pandas as pd

# Load Euro 2012 dataset
csv_path = "Euro_2012_stats_TEAM.csv"
if not os.path.exists(csv_path):
    csv_path = "https://raw.githubusercontent.com/guipsamora/pandas_exercises/master/02_Filtering_%26_Sorting/Euro12/Euro_2012_stats_TEAM.csv"

euro12 = pd.read_csv(csv_path)
print(f"Euro 2012 dataset loaded. Dimensions: {euro12.shape}")
euro12.head(3)

Euro 2012 dataset loaded. Dimensions: (16, 35)


,Team,Goals,Shots on target,Shots off target,Shooting Accuracy,% Goals-to-shots,Total shots (inc. Blocked),Hit Woodwork,Penalty goals,Penalties not scored,...,Saves made,Saves-to-shots ratio,Fouls Won,Fouls Conceded,Offsides,Yellow Cards,Red Cards,Subs on,Subs off,Players Used
0,Croatia,4,13,12,51.9%,16.0%,32,0,0,0,...,13,81.3%,41,62,2,9,0,9,9,16
1,Czech Republic,4,13,18,41.9%,12.9%,39,0,0,0,...,9,60.1%,53,73,8,7,0,11,11,19
2,Denmark,4,10,10,50.0%,20.0%,27,1,0,0,...,10,66.7%,25,38,8,4,0,7,7,15


## 2. Feature Selection & Tournament Metrics

In [2]:
# Goal scoring Series
euro12["Goals"].head()

0    4
1    4
2    4
3    5
4    3
Name: Goals, dtype: int64

In [3]:
# Total participating teams
num_teams = euro12["Team"].nunique()
print(f"Number of participating teams: {num_teams}")

Number of participating teams: 16


In [4]:
# Total number of features (columns)
print(f"Number of columns: {euro12.shape[1]}")

Number of columns: 35


## 3. Disciplinary Analysis: Sorting & Averages

In [5]:
# Isolate discipline columns
discipline = euro12[["Team", "Yellow Cards", "Red Cards"]].copy()
discipline.head(4)

,Team,Yellow Cards,Red Cards
0,Croatia,9,0
1,Czech Republic,7,0
2,Denmark,4,0
3,England,5,0


In [6]:
# Sort teams by Red Cards (descending), then Yellow Cards (descending)
discipline_sorted = discipline.sort_values(
    by=["Red Cards", "Yellow Cards"], 
    ascending=[False, False]
)
discipline_sorted.head(6)

,Team,Yellow Cards,Red Cards
6,Greece,9,1
9,Poland,7,1
11,Republic of Ireland,6,1
7,Italy,16,0
10,Portugal,12,0
13,Spain,11,0


In [7]:
# Mean yellow cards per team
mean_yellow = discipline["Yellow Cards"].mean()
print(f"Mean Yellow Cards per team: {mean_yellow:.2f}")

Mean Yellow Cards per team: 7.44


## 4. Performance & String Filtering

In [8]:
# High-scoring teams (> 6 goals)
high_scorers = euro12[euro12["Goals"] > 6]
high_scorers[["Team", "Goals", "Shooting Accuracy"]]

,Team,Goals,Shooting Accuracy
5,Germany,10,47.8%
13,Spain,12,55.9%


In [9]:
# Teams whose name starts with the letter 'G'
g_teams = euro12[euro12["Team"].str.startswith("G")]
g_teams[["Team", "Goals"]]

,Team,Goals
5,Germany,10
6,Greece,5


## 5. Positional 2D Slicing with `.iloc`

### ⚠️ Top Interview Question: Pythonic Negative Slicing in DataFrames
- First 7 columns: `euro12.iloc[:, 0:7]` (columns 0 through 6).
- All columns except the last 3: `euro12.iloc[:, :-3]`.
  This is identical to Python slice mechanics `list[:-3]`.

In [10]:
# Select the first 7 columns across all rows
euro12.iloc[:, 0:7].head(3)

,Team,Goals,Shots on target,Shots off target,Shooting Accuracy,% Goals-to-shots,Total shots (inc. Blocked)
0,Croatia,4,13,12,51.9%,16.0%,32
1,Czech Republic,4,13,18,41.9%,12.9%,39
2,Denmark,4,10,10,50.0%,20.0%,27


In [11]:
# Select all columns except the last 3
euro12.iloc[:, :-3].head(3)

,Team,Goals,Shots on target,Shots off target,Shooting Accuracy,% Goals-to-shots,Total shots (inc. Blocked),Hit Woodwork,Penalty goals,Penalties not scored,...,Clean Sheets,Blocks,Goals conceded,Saves made,Saves-to-shots ratio,Fouls Won,Fouls Conceded,Offsides,Yellow Cards,Red Cards
0,Croatia,4,13,12,51.9%,16.0%,32,0,0,0,...,0,10,3,13,81.3%,41,62,2,9,0
1,Czech Republic,4,13,18,41.9%,12.9%,39,0,0,0,...,1,10,6,9,60.1%,53,73,8,7,0
2,Denmark,4,10,10,50.0%,20.0%,27,1,0,0,...,1,10,5,10,66.7%,25,38,8,4,0


## 6. Targeted Team Lookups: Fixing the Hardcoded Index Bug

In the original exercise, step 14 asked:
> *"Present only the Shooting Accuracy from England, Italy and Russia"*

The original notebook contained:
```python
euro12.loc[3, "Shooting Accuracy"]  # Only returned England because it was at row index 3!
```

### ⚠️ Why Hardcoded Integer Row Labels Fail:
- Row 3 was England *only* because the CSV happened to be in that order. If the DataFrame was sorted by Goals, row 3 is no longer England!
- It failed to include Italy and Russia entirely.
- **The Idiomatic Solution**: Use `.isin()` or set `'Team'` as the index!

In [12]:
# ✅ IDIOMATIC SOLUTION: Filtering with .isin()
target_countries = ["England", "Italy", "Russia"]
shooting_accuracy = euro12.loc[
    euro12["Team"].isin(target_countries), 
    ["Team", "Shooting Accuracy"]
].reset_index(drop=True)

display(shooting_accuracy)

,Team,Shooting Accuracy
0,England,50.0%
1,Italy,43.0%
2,Russia,22.5%


In [13]:
# Alternative: Label lookup with Team as Index
accuracy_indexed = euro12.set_index("Team").loc[target_countries, ["Shooting Accuracy"]]
display(accuracy_indexed)

,Shooting Accuracy
Team,
England,50.0%
Italy,43.0%
Russia,22.5%


## 7. Multi-Axis Slicing Cheat Sheet

| Task | Idiomatic Expression | Key Notes |
| :--- | :--- | :--- |
| **All except last K cols** | `df.iloc[:, :-K]` | Clean negative slice |
| **First N cols** | `df.iloc[:, :N]` | Endpoint excluded |
| **Multi-Value Lookup** | `df.loc[df['col'].isin([...]), ['cols']]` | Robust against row reordering |
| **Prefix Filtering** | `df['col'].str.startswith('X')` | Case-sensitive |
| **Multi-Sort Priority** | `df.sort_values(by=['A', 'B'], ascending=[False, True])` | Priority order matches list order |

---
## 🎯 8. Technical Interview Corner: Tricky Questions & Drills

### Q1: `df.iloc[:, :-3]` vs `df.drop(columns=df.columns[-3:])`
**Question**: What is the difference between slicing with `iloc[:, :-3]` versus dropping with `.drop()`?

**Answer**:
- `df.iloc[:, :-3]`: Slices the column axis directly. It returns a view or lightweight slice without creating unnecessary intermediate column arrays.
- `df.drop(columns=...)`: Requires evaluating `df.columns[-3:]`, generating a list of names, and constructing a new DataFrame with those columns omitted.
- *Best Practice*: For simple positional truncation, `iloc[:, :-K]` is much more concise and idiomatic.

### Q2: Converting Percentage Strings to Floats
**Question**: Notice that `Shooting Accuracy` is stored as an object string with a percent sign (e.g. `'51.9%'`). How would you convert this into a pure numeric float Series in a vectorized fashion?

**Answer**:
Strip the `%` character using `.str.rstrip('%')` and cast to `float` divided by 100.

In [14]:
# Vectorized percentage conversion
euro12["Shooting Accuracy (Float)"] = (
    euro12["Shooting Accuracy"].str.rstrip("%").astype(float) / 100
)

euro12[["Team", "Shooting Accuracy", "Shooting Accuracy (Float)"]].head(4)

,Team,Shooting Accuracy,Shooting Accuracy (Float)
0,Croatia,51.9%,0.519
1,Czech Republic,41.9%,0.419
2,Denmark,50.0%,0.500
3,England,50.0%,0.500


### Q3: Advanced Interview Challenge: Most Efficient Attacking Teams
**Challenge**: In a single chained expression, find the top 3 teams with the highest **Goals per Shot on Target**, considering only teams that took **at least 15 shots on target**!

In [15]:
# Solution to Coding Challenge
shot_efficiency = (
    euro12[euro12["Shots on target"] >= 15]
    .assign(goal_to_shot_ratio=lambda df: (df["Goals"] / df["Shots on target"]).round(3))
    .sort_values(by="goal_to_shot_ratio", ascending=False)
    [["Team", "Goals", "Shots on target", "goal_to_shot_ratio"]]
    .head(3)
    .reset_index(drop=True)
)

shot_efficiency

,Team,Goals,Shots on target,goal_to_shot_ratio
0,Germany,10,32,0.312
1,Sweden,5,17,0.294
2,Spain,12,42,0.286
